# 05 — CLIP Zero-Shot Classification and BLIP Captioning

**Project:** Brain MRI Image Classification and Explainable AI  
**Notebook goal:** add two *exploratory, frozen-model* vision-language experiments after completing notebooks 01–04:

1. **CLIP zero-shot classification:** compare prompt-based image/text similarity against the supervised CNN, ResNet-18, and ViT models.
2. **BLIP image captioning:** generate captions for a small, class-balanced sample and review them qualitatively.

## Outputs

- `results/clip_metrics.json` — model, prompts, split counts and CLIP metrics
- `results/clip_classification_report.csv` — per-class metrics on the official test set
- `results/clip_predictions.csv` — image-level CLIP predictions and probabilities
- `results/model_comparison.csv` — updated comparison rows for CLIP, alongside existing models when available
- `results/blip_captions.csv` — BLIP captions for a small balanced sample
- `results/visualizations/clip_confusion_matrix_*.png` and `blip_caption_examples.png`

## Important interpretation limits

- This notebook does **not train or fine-tune** CLIP or BLIP. CLIP scores zero-shot candidate labels; BLIP generates captions.
- The selected prompt templates are a starting point, not medically validated prompts. Do not tune prompts on the test set.
- General-purpose CLIP/BLIP checkpoints are not validated for MRI diagnosis. Caption text can be incorrect or hallucinated. Treat outputs as an educational experiment, not clinical evidence or medical advice.
- The data may contain duplicate/near-duplicate images, source artifacts, and no reliable patient identifiers. The duplicate-filtered evaluation is a sensitivity analysis, not a patient-level independent test.

## Prerequisites

Run notebooks `01_eda_and_data_audit.ipynb` and `02_custom_cnn.ipynb` first so `results/model_data_split.csv` exists. Notebooks 03 and 04 are recommended for the complete supervised comparison. Install the project's requirements before running this notebook. The first run may download model weights from Hugging Face and needs internet access.

## 1. Imports and reproducibility

If `transformers` is not installed, run `pip install -r requirements.txt` from the project root and restart the kernel if needed.

In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from IPython.display import display

import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm
from transformers import (
    CLIPModel,
    CLIPProcessor,
    BlipForConditionalGeneration,
    BlipProcessor,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Transformers imports successful")

## 2. Configuration and output paths

Start with CLIP evaluation on the full official test split and BLIP captioning on a small sample. On a CPU-only machine, CLIP may take a while; set `RUN_BLIP = False` to skip caption generation until later.

In [ ]:
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
DATA_BASE = PROJECT_ROOT / "data"
DATASET_ROOT_OVERRIDE = None  # Optional: Path(r"D:/datasets/brain-tumor-mri-dataset")
RESULTS_DIR = PROJECT_ROOT / "results"
VIS_DIR = RESULTS_DIR / "visualizations"
MODELS_DIR = PROJECT_ROOT / "models"
for folder in (RESULTS_DIR, VIS_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SPLIT_PATH = RESULTS_DIR / "model_data_split.csv"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
BLIP_MODEL_NAME = "Salesforce/blip-image-captioning-base"
RUN_CLIP = True
RUN_BLIP = True
BATCH_SIZE = 32 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0  # Kept for consistency; PIL-based inference is batched explicitly below.
BLIP_IMAGES_PER_CLASS = 3  # At most 12 examples with the standard four classes.
BLIP_MAX_NEW_TOKENS = 30

CLIP_METRICS_PATH = RESULTS_DIR / "clip_metrics.json"
CLIP_REPORT_PATH = RESULTS_DIR / "clip_classification_report.csv"
CLIP_PREDICTIONS_PATH = RESULTS_DIR / "clip_predictions.csv"
BLIP_CAPTIONS_PATH = RESULTS_DIR / "blip_captions.csv"
COMPARISON_PATH = RESULTS_DIR / "model_comparison.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset search root:", DATA_BASE)
print("Shared split manifest:", SPLIT_PATH)
print("CLIP model:", CLIP_MODEL_NAME)
print("BLIP model:", BLIP_MODEL_NAME)

## 3. Load the shared split manifest

This notebook does not create new splits. It uses the exact manifest from notebook 02 so the supervised model comparison remains aligned. The secondary test subset removes images whose exact hashes occur anywhere in the original training data (the train and validation rows in the manifest).

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
TRAIN_NAMES = {"training", "train"}
TEST_NAMES = {"testing", "test"}


def class_dirs_with_images(split_dir: Path):
    if not split_dir.is_dir():
        return []
    return [
        child for child in sorted(split_dir.iterdir())
        if child.is_dir() and any(
            p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
            for p in child.rglob("*")
        )
    ]


def find_dataset_root(search_root: Path) -> Path:
    if not search_root.exists():
        raise FileNotFoundError(
            f"Dataset root not found: {search_root}. Download/extract the Kaggle dataset under data/ "
            "or update DATA_BASE / DATASET_ROOT_OVERRIDE in the configuration cell."
        )
    candidates = [p for p in search_root.rglob("*") if p.is_dir()]
    candidates.append(search_root)
    train_dirs = [p for p in candidates if p.name.lower() in TRAIN_NAMES and class_dirs_with_images(p)]
    test_dirs = [p for p in candidates if p.name.lower() in TEST_NAMES and class_dirs_with_images(p)]
    pairs = [(tr, te) for tr in train_dirs for te in test_dirs if tr.parent == te.parent]
    if not pairs:
        raise FileNotFoundError(
            f"Could not find matching Training/Testing or train/test directories under {search_root}. "
            "Check the folder layout described in README.md."
        )
    pairs.sort(key=lambda pair: len(pair[0].parts) + len(pair[1].parts))
    return pairs[0][0].parent


if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SPLIT_PATH}. Run notebook 02_custom_cnn.ipynb first; it creates the shared split manifest."
    )

DATASET_ROOT = Path(DATASET_ROOT_OVERRIDE).expanduser().resolve() if DATASET_ROOT_OVERRIDE else find_dataset_root(DATA_BASE)
split_df = pd.read_csv(SPLIT_PATH)
required_columns = {"model_split", "class_name", "relative_path", "sha256"}
missing_columns = required_columns - set(split_df.columns)
if missing_columns:
    raise ValueError(f"Split manifest is missing required columns: {sorted(missing_columns)}")

split_df["model_split"] = split_df["model_split"].astype(str).str.lower().str.strip()
split_df["class_name"] = split_df["class_name"].astype(str).str.lower().str.strip()
split_df["relative_path"] = split_df["relative_path"].astype(str)
split_df["sha256"] = split_df["sha256"].astype(str)
CLASS_NAMES = sorted(split_df["class_name"].dropna().unique().tolist(), key=str.casefold)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
split_df["path"] = split_df["relative_path"].map(lambda rel: str((DATASET_ROOT / Path(rel)).resolve()))

missing_paths = split_df.loc[~split_df["path"].map(lambda value: Path(value).is_file()), "relative_path"].tolist()
if missing_paths:
    raise FileNotFoundError(
        f"{len(missing_paths)} manifest paths were not found beneath {DATASET_ROOT}. "
        f"Examples: {missing_paths[:5]}. Correct DATASET_ROOT_OVERRIDE if the dataset moved."
    )

if not {"train", "validation", "test"}.issubset(set(split_df["model_split"])):
    raise ValueError("The shared manifest must contain train, validation, and test rows.")
if len(CLASS_NAMES) < 2:
    raise ValueError(f"Expected at least two classes; found {CLASS_NAMES}")

train_val_df = split_df.loc[split_df["model_split"].isin(["train", "validation"])].copy()
official_test_df = split_df.loc[split_df["model_split"] == "test"].copy().reset_index(drop=True)
train_val_hashes = set(train_val_df["sha256"].dropna().astype(str))
test_exact_filtered_df = official_test_df.loc[
    ~official_test_df["sha256"].astype(str).isin(train_val_hashes)
].copy().reset_index(drop=True)

print("Dataset root:", DATASET_ROOT)
print("Class order:", CLASS_NAMES)
print("Class mapping:", CLASS_TO_IDX)
print("Train:", int((split_df["model_split"] == "train").sum()))
print("Validation:", int((split_df["model_split"] == "validation").sum()))
print("Official test:", len(official_test_df))
print("Exact-duplicate-filtered test:", len(test_exact_filtered_df))
display(split_df.groupby(["model_split", "class_name"]).size().rename("images").reset_index())

## 4. Define class names and CLIP prompt ensemble

The notebook uses several simple prompts per class and averages CLIP's similarity logits across prompts belonging to that class. Prompt templates are fixed before test evaluation. If you later experiment with different prompts, document that as a separate exploratory run rather than repeatedly tuning against the test labels.

In [ ]:
# Human-readable labels aligned with the folder names used in this Kaggle dataset.
KNOWN_DISPLAY_NAMES = {
    "glioma": "glioma tumor",
    "meningioma": "meningioma tumor",
    "notumor": "no tumor",
    "no_tumor": "no tumor",
    "pituitary": "pituitary tumor",
}
CLASS_DISPLAY_NAMES = {
    name: KNOWN_DISPLAY_NAMES.get(name, name.replace("_", " ").replace("-", " "))
    for name in CLASS_NAMES
}

PROMPT_TEMPLATES = [
    "an MRI scan of a brain showing {}.",
    "a brain MRI image with {}.",
    "a medical MRI image of {}.",
    "an axial brain MRI showing {}.",
]

PROMPTS_BY_CLASS = {
    name: [template.format(CLASS_DISPLAY_NAMES[name]) for template in PROMPT_TEMPLATES]
    for name in CLASS_NAMES
}
ALL_PROMPTS = [prompt for name in CLASS_NAMES for prompt in PROMPTS_BY_CLASS[name]]
PROMPT_CLASS_INDICES = np.repeat(np.arange(len(CLASS_NAMES)), len(PROMPT_TEMPLATES))

prompt_preview = pd.DataFrame({
    "class_name": [name for name in CLASS_NAMES for _ in PROMPT_TEMPLATES],
    "prompt": ALL_PROMPTS,
})
display(prompt_preview)
print("These prompts are fixed for this run. They have not been medically validated.")

## 5. CLIP zero-shot inference helpers

CLIP is frozen: there is no backpropagation or training loop. The model compares each MRI image to all candidate prompts; we average prompt scores within each class and apply softmax across class scores to produce relative candidate probabilities. These probabilities are not calibrated clinical probabilities.

In [ ]:
def safe_multiclass_roc_auc(y_true, y_prob, n_classes):
    """Return macro one-vs-rest ROC-AUC only when every class appears in y_true."""
    y_true = np.asarray(y_true, dtype=int)
    if len(np.unique(y_true)) != n_classes:
        return None
    try:
        return float(roc_auc_score(
            y_true,
            np.asarray(y_prob),
            labels=list(range(n_classes)),
            multi_class="ovr",
            average="macro",
        ))
    except (ValueError, TypeError):
        return None


def compute_metrics(y_true, y_pred, y_prob, elapsed_forward_seconds, n_classes):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    n_images = int(len(y_true))
    if n_images == 0:
        return None
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "roc_auc_ovr_macro": safe_multiclass_roc_auc(y_true, y_prob, n_classes),
        "forward_latency_ms_per_image": float(elapsed_forward_seconds * 1000.0 / n_images),
        "num_images": n_images,
    }


def save_confusion_matrix(y_true, y_pred, class_names, title, output_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)), class_names, rotation=35, ha="right")
    ax.set_yticks(range(len(class_names)), class_names)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, str(cm[row, col]), ha="center", va="center")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", output_path)


def load_rgb_images(batch_frame):
    images = []
    for row in batch_frame.itertuples(index=False):
        image_path = Path(row.path)
        try:
            with Image.open(image_path) as image_obj:
                images.append(image_obj.convert("RGB"))
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            raise RuntimeError(f"Could not read image {image_path}: {exc}") from exc
    return images


def run_clip_evaluation(frame, evaluation_name, clip_model, clip_processor, text_inputs):
    """Evaluate a dataframe of manifest rows without fitting or prompt tuning."""
    if frame.empty:
        print(f"Skipping {evaluation_name}: subset is empty.")
        return None, None

    frame = frame.reset_index(drop=True).copy()
    y_true, y_pred, y_prob = [], [], []
    all_forward_seconds = 0.0
    class_count = len(CLASS_NAMES)
    template_count = len(PROMPT_TEMPLATES)
    clip_model.eval()

    for start in tqdm(range(0, len(frame), BATCH_SIZE), desc=f"CLIP: {evaluation_name}"):
        batch_frame = frame.iloc[start:start + BATCH_SIZE]
        images = load_rgb_images(batch_frame)
        image_inputs = clip_processor(images=images, return_tensors="pt")
        pixel_values = image_inputs["pixel_values"].to(DEVICE)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        tic = time.perf_counter()
        with torch.inference_mode():
            outputs = clip_model(
                pixel_values=pixel_values,
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs.get("attention_mask"),
            )
            prompt_logits = outputs.logits_per_image
            class_logits = prompt_logits.reshape(
                len(batch_frame), class_count, template_count
            ).mean(dim=2)
            probabilities = torch.softmax(class_logits, dim=1)
            predictions = probabilities.argmax(dim=1)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        all_forward_seconds += time.perf_counter() - tic

        y_true.extend(batch_frame["class_name"].map(CLASS_TO_IDX).astype(int).tolist())
        y_pred.extend(predictions.cpu().numpy().astype(int).tolist())
        y_prob.extend(probabilities.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred, y_prob, all_forward_seconds, class_count)
    prediction_df = frame.copy()
    prediction_df["evaluation"] = evaluation_name
    prediction_df["true_index"] = y_true
    prediction_df["predicted_index"] = y_pred
    prediction_df["true_class"] = [CLASS_NAMES[i] for i in y_true]
    prediction_df["predicted_class"] = [CLASS_NAMES[i] for i in y_pred]
    prediction_df["confidence"] = [float(row[pred]) for row, pred in zip(y_prob, y_pred)]
    for idx, class_name in enumerate(CLASS_NAMES):
        safe_class_name = "".join(ch if ch.isalnum() else "_" for ch in class_name)
        prediction_df[f"probability__{safe_class_name}"] = [float(row[idx]) for row in y_prob]

    prediction_df.to_csv(
        RESULTS_DIR / f"clip_predictions_{evaluation_name}.csv", index=False
    )
    return metrics, prediction_df

## 6. Load CLIP and tokenize prompts

The first run downloads the pretrained checkpoint and processor. This may take time depending on network speed and cache state.

In [ ]:
clip_model = None
clip_processor = None
text_inputs = None
clip_official_metrics = None
clip_filtered_metrics = None
clip_prediction_parts = []
clip_classification_report_df = None

if RUN_CLIP:
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
    clip_model.eval()
    text_inputs = clip_processor(
        text=ALL_PROMPTS,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    text_inputs = {key: value.to(DEVICE) for key, value in text_inputs.items()}
    print("CLIP loaded:", CLIP_MODEL_NAME)
    print("Number of prompts:", len(ALL_PROMPTS))
else:
    print("RUN_CLIP=False; CLIP inference will be skipped.")

## 7. Evaluate CLIP on the official and duplicate-filtered test sets

The official test metrics and the duplicate-filtered sensitivity analysis are reported separately. Neither set is used to train CLIP or select the prompts.

In [ ]:
if RUN_CLIP:
    clip_official_metrics, official_clip_predictions = run_clip_evaluation(
        official_test_df, "official_test", clip_model, clip_processor, text_inputs
    )
    if official_clip_predictions is not None:
        clip_prediction_parts.append(official_clip_predictions)
        y_true = official_clip_predictions["true_index"].to_numpy(dtype=int)
        y_pred = official_clip_predictions["predicted_index"].to_numpy(dtype=int)
        prob_cols = [
            f"probability__{''.join(ch if ch.isalnum() else '_' for ch in name)}"
            for name in CLASS_NAMES
        ]
        y_prob = official_clip_predictions[prob_cols].to_numpy(dtype=float)
        clip_classification_report_df = pd.DataFrame(classification_report(
            y_true, y_pred,
            labels=list(range(len(CLASS_NAMES))),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )).T
        clip_classification_report_df.to_csv(CLIP_REPORT_PATH, index=True)
        display(clip_classification_report_df.round(4))
        save_confusion_matrix(
            y_true, y_pred, CLASS_NAMES,
            "CLIP Zero-Shot — Official Test Confusion Matrix",
            VIS_DIR / "clip_confusion_matrix_official_test.png",
        )

    clip_filtered_metrics, filtered_clip_predictions = run_clip_evaluation(
        test_exact_filtered_df, "test_exact_duplicate_filtered", clip_model, clip_processor, text_inputs
    )
    if filtered_clip_predictions is not None:
        clip_prediction_parts.append(filtered_clip_predictions)
        save_confusion_matrix(
            filtered_clip_predictions["true_index"].to_numpy(dtype=int),
            filtered_clip_predictions["predicted_index"].to_numpy(dtype=int),
            CLASS_NAMES,
            "CLIP Zero-Shot — Exact-Duplicate-Filtered Test Confusion Matrix",
            VIS_DIR / "clip_confusion_matrix_exact_filtered_test.png",
        )

    if clip_prediction_parts:
        pd.concat(clip_prediction_parts, ignore_index=True).to_csv(
            CLIP_PREDICTIONS_PATH, index=False
        )
        print("Saved CLIP predictions:", CLIP_PREDICTIONS_PATH)
        display(pd.DataFrame([
            {"evaluation": "official_test", **(clip_official_metrics or {})},
            {"evaluation": "test_exact_duplicate_filtered", **(clip_filtered_metrics or {})},
        ]).round(4))

## 8. Update the shared model comparison table

This adds CLIP as a zero-shot row while preserving metrics already written by notebooks 03 and 04. CLIP is not a trained model, so the comparison is descriptive rather than a perfectly controlled architecture study: the supervised models were fine-tuned on the dataset, while CLIP uses text prompts only.

In [ ]:
if RUN_CLIP:
    metric_keys = [
        "accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "forward_latency_ms_per_image", "num_images",
    ]
    comparison_rows = []
    if COMPARISON_PATH.exists():
        try:
            previous = pd.read_csv(COMPARISON_PATH)
            if {"model", "evaluation"}.issubset(previous.columns):
                previous = previous.loc[
                    previous["model"].astype(str).str.lower() != "clip zero-shot"
                ]
                comparison_rows.extend(previous.to_dict(orient="records"))
            else:
                warnings.warn("Existing comparison table has an unexpected schema; rebuilding with CLIP rows.")
        except (OSError, pd.errors.ParserError) as exc:
            warnings.warn(f"Could not read previous comparison table: {exc}; rebuilding with CLIP rows. ")

    if clip_official_metrics is not None:
        comparison_rows.append({
            "model": "CLIP zero-shot",
            "evaluation": "official_test",
            **{key: clip_official_metrics.get(key) for key in metric_keys},
        })
    if clip_filtered_metrics is not None:
        comparison_rows.append({
            "model": "CLIP zero-shot",
            "evaluation": "test_exact_duplicate_filtered",
            **{key: clip_filtered_metrics.get(key) for key in metric_keys},
        })
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(COMPARISON_PATH, index=False)
    print("Updated model comparison:", COMPARISON_PATH)
    display(comparison_df.round(4))

## 9. Save CLIP experiment summary

In [ ]:
def json_safe(value):
    """Convert NumPy scalars and undefined metric values to JSON-safe objects."""
    if isinstance(value, dict):
        return {str(key): json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(val) for val in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value

if RUN_CLIP:
    clip_summary = {
        "experiment": "CLIP zero-shot image classification",
        "model_name": CLIP_MODEL_NAME,
        "trained_on_dataset": False,
        "class_names": CLASS_NAMES,
        "class_display_names": CLASS_DISPLAY_NAMES,
        "prompt_templates": PROMPT_TEMPLATES,
        "prompts_by_class": PROMPTS_BY_CLASS,
        "batch_size": BATCH_SIZE,
        "device": str(DEVICE),
        "split_manifest": str(SPLIT_PATH.relative_to(PROJECT_ROOT)),
        "split_sizes": {
            "train": int((split_df["model_split"] == "train").sum()),
            "validation": int((split_df["model_split"] == "validation").sum()),
            "official_test": int(len(official_test_df)),
            "test_exact_duplicate_filtered": int(len(test_exact_filtered_df)),
        },
        "official_test_metrics": clip_official_metrics,
        "exact_duplicate_filtered_test_metrics": clip_filtered_metrics,
        "limitations": [
            "Frozen general-purpose CLIP model; no fine-tuning on this dataset.",
            "Prompt templates are exploratory and not medically validated.",
            "Softmax values are relative candidate scores, not calibrated clinical probabilities.",
            "Duplicate-filtered evaluation removes exact hashes found in train/validation; it does not ensure patient-level independence.",
            "This experiment is educational only and is not a clinical diagnostic tool.",
        ],
    }
    CLIP_METRICS_PATH.write_text(
        json.dumps(json_safe(clip_summary), indent=2, allow_nan=False), encoding="utf-8"
    )
    print("Saved CLIP summary:", CLIP_METRICS_PATH)

## 10. Free memory before loading BLIP

CLIP and BLIP are separate models. On a GPU, delete the CLIP objects before loading BLIP to reduce memory pressure.

In [ ]:
if clip_model is not None:
    del clip_model
if text_inputs is not None:
    del text_inputs
if clip_processor is not None:
    del clip_processor
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released CLIP model objects before the BLIP experiment.")

## 11. Select a small, class-balanced BLIP caption sample

BLIP captioning is qualitative only. To make the review manageable, select up to `BLIP_IMAGES_PER_CLASS` examples per label from the official test set. The ground-truth label is saved for later human review but is **not passed to the captioning model**.

In [ ]:
blip_sample_df = (
    official_test_df.groupby("class_name", group_keys=False, sort=True)
    .apply(lambda group: group.sample(
        n=min(BLIP_IMAGES_PER_CLASS, len(group)), random_state=SEED
    ))
    .reset_index(drop=True)
)
if len(blip_sample_df) > 0:
    display(blip_sample_df[["relative_path", "class_name", "sha256"]])
else:
    print("No official test rows available for BLIP captioning.")
print("BLIP sample size:", len(blip_sample_df))

## 12. Generate BLIP captions (optional)

BLIP is a general-purpose captioning checkpoint, not an MRI-specialist model. It may describe the image incorrectly, produce generic captions, or fail to identify the relevant anatomy. Read the outputs critically; do not use them as diagnoses or as labels for training.

In [ ]:
blip_caption_df = pd.DataFrame()

if RUN_BLIP and not blip_sample_df.empty:
    blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_NAME)
    blip_model = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL_NAME).to(DEVICE)
    blip_model.eval()
    caption_rows = []

    for row in tqdm(blip_sample_df.itertuples(index=False), total=len(blip_sample_df), desc="BLIP captioning"):
        image_path = Path(row.path)
        with Image.open(image_path) as image_obj:
            image = image_obj.convert("RGB")
        inputs = blip_processor(images=image, return_tensors="pt")
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        with torch.inference_mode():
            output_ids = blip_model.generate(
                **inputs,
                max_new_tokens=BLIP_MAX_NEW_TOKENS,
                num_beams=3,
                do_sample=False,
            )
        caption = blip_processor.decode(output_ids[0], skip_special_tokens=True).strip()
        caption_rows.append({
            "relative_path": row.relative_path,
            "sha256": row.sha256,
            "dataset_label_for_review_only": row.class_name,
            "caption": caption,
            "model_name": BLIP_MODEL_NAME,
        })

    blip_caption_df = pd.DataFrame(caption_rows)
    blip_caption_df.to_csv(BLIP_CAPTIONS_PATH, index=False)
    display(blip_caption_df)
    print("Saved BLIP captions:", BLIP_CAPTIONS_PATH)

    # Contact sheet for qualitative review. The true label is shown for human review only.
    n = len(blip_caption_df)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.6 * nrows))
    axes = np.array(axes).reshape(-1)
    caption_lookup = {
        row["relative_path"]: row for row in blip_caption_df.to_dict(orient="records")
    }
    sample_lookup = {
        row["relative_path"]: row for row in blip_sample_df.to_dict(orient="records")
    }
    for ax, relative_path in zip(axes, blip_caption_df["relative_path"].tolist()):
        meta = caption_lookup[relative_path]
        sample_meta = sample_lookup[relative_path]
        with Image.open(sample_meta["path"]) as image_obj:
            image = image_obj.convert("RGB")
        ax.imshow(image, cmap="gray")
        ax.axis("off")
        ax.set_title(
            f"Dataset label (review only): {meta['dataset_label_for_review_only']}\n"
            f"BLIP caption: {meta['caption']}",
            fontsize=9,
            wrap=True,
        )
    for ax in axes[len(blip_caption_df):]:
        ax.axis("off")
    fig.suptitle("BLIP captions — qualitative review, not diagnostic output", fontsize=14)
    fig.tight_layout()
    caption_grid_path = VIS_DIR / "blip_caption_examples.png"
    fig.savefig(caption_grid_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved caption review grid:", caption_grid_path)

    del blip_model, blip_processor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif not RUN_BLIP:
    print("RUN_BLIP=False; captioning skipped. Set it to True to run the experiment later.")
else:
    print("BLIP captioning skipped because no test samples were available.")

## 13. Interpret the CLIP and BLIP results

Review these questions after the cells above complete:

1. How does CLIP zero-shot macro-F1 compare with the supervised models in `results/model_comparison.csv`?
2. Which classes are most frequently confused in the CLIP confusion matrix?
3. Does the duplicate-filtered test result differ noticeably from the official test result?
4. Do BLIP captions describe image content meaningfully, or do they produce generic or incorrect text?
5. Why should a general-purpose vision-language model's score or caption not be treated as a medical diagnosis?

CLIP is a zero-shot baseline and BLIP captions are qualitative outputs. Do not tune prompts, discard difficult test examples, or change evaluation choices based on the test labels. If you revise prompts, record the change and use validation data for development; retain the held-out test for one final evaluation.

## 14. Completion checklist

- [ ] Reused `results/model_data_split.csv`; did not create a new split.
- [ ] Recorded the exact CLIP and BLIP checkpoint names and the prompt templates used.
- [ ] Saved CLIP official-test predictions, classification report, metrics, and confusion matrix.
- [ ] Reported the exact-duplicate-filtered test result separately when the subset was non-empty.
- [ ] Reviewed BLIP captions manually and documented that they may be inaccurate on MRI images.
- [ ] Preserved the existing CNN, ResNet-18, and ViT rows in `results/model_comparison.csv`.
- [ ] Did not describe CLIP/BLIP results as clinically validated or add unmeasured claims to the resume.

### Project milestone

At this point, the planned model experiments are covered: custom CNN, ResNet-18 transfer learning, ViT-B/16 fine-tuning, CLIP zero-shot classification, and optional BLIP captioning. The next engineering milestone is `app.py`: a small Gradio interface for the **supervised classifier selected from validation results**, with clearly stated educational limitations.